# KV Cache Hierarchical Memory Manager – Demo

SRAM/HBM 2-tier KV cache management with cosine-similarity importance scoring.

**Run this on Google Colab (A100 recommended) or locally with a GPU.**

In [ ]:
!pip install -q torch transformers sentence-transformers datasets tqdm

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()

print(f"Loaded {model_name}: {model.config.n_layer} layers, {model.config.n_embd} dim")

In [ ]:
from kv_cache_manager import CacheManager, ImportanceScorer, ManagedGenerator

scorer = ImportanceScorer(sim_threshold=0.30, boost=0.35, decay=0.04, promote_thresh=0.68)
cache = CacheManager(sram_capacity=128, dim=model.config.n_embd, device=device, scorer=scorer)
gen = ManagedGenerator(model, tokenizer, cache)

## 1. Text Generation with Managed KV Cache

In [ ]:
prompt = "The history of artificial intelligence began"
output = gen.generate(prompt, max_new_tokens=80, temperature=0.8)
print(f"Prompt: {prompt}")
print(f"Generated: {output}")
print()

stats = cache.stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 2. SRAM / HBM Tier Inspection

In [ ]:
def show_tier(label, entries, tokenizer, n=15):
    sorted_e = sorted(entries, key=lambda e: e.importance, reverse=True)
    print(f"\n{label} ({len(entries)} tokens, showing top {min(n, len(entries))})")
    print(f"{'pos':>5}  {'imp':>6}  {'refs':>4}  token")
    print("-" * 40)
    for e in sorted_e[:n]:
        tok = tokenizer.decode([e.token_id]).replace('\n', '\\n')
        bar = '█' * int(e.importance * 20)
        print(f"{e.position:>5}  {e.importance:>6.3f}  {e.refs:>4}  {bar}  '{tok}'")

show_tier("SRAM", cache.sram, tokenizer)
show_tier("HBM", cache.hbm, tokenizer)

## 3. Perplexity Comparison: Managed vs Full Attention

In [ ]:
sample_text = (
    "The history of artificial intelligence began in antiquity, with myths, "
    "stories and rumors of artificial beings endowed with intelligence or "
    "consciousness by master craftsmen. The seeds of modern AI were planted by "
    "philosophers who attempted to describe the process of human thinking as "
    "the mechanical manipulation of symbols. This work culminated in the "
    "invention of the programmable digital computer in the 1940s, a machine "
    "based on the abstract essence of mathematical reasoning."
)

result = gen.evaluate_perplexity(sample_text, max_length=256)

print(f"Sequence length : {result['sequence_length']}")
print(f"Baseline PPL    : {result['baseline_ppl']:.2f}")
print(f"Managed PPL     : {result['managed_ppl']:.2f}")
print(f"PPL ratio       : {result['ppl_ratio']:.4f}  (target ≤ 1.05)")
print()
for k, v in result['cache_stats'].items():
    print(f"  {k}: {v}")

## 4. Hyperparameter Sweep

In [ ]:
results = []
for sram_cap in [32, 64, 128, 256]:
    cache_sweep = CacheManager(sram_capacity=sram_cap, dim=model.config.n_embd, device=device)
    gen_sweep = ManagedGenerator(model, tokenizer, cache_sweep)
    r = gen_sweep.evaluate_perplexity(sample_text, max_length=256)
    results.append((sram_cap, r))
    print(f"sram={sram_cap:>4d}  baseline={r['baseline_ppl']:.2f}  managed={r['managed_ppl']:.2f}  ratio={r['ppl_ratio']:.4f}")

print("\n→ Higher SRAM capacity → ratio closer to 1.0 (lossless)")